# Práctica — Clase 7: NLP II, sentimiento, RNN, LSTM y generación de texto

En esta práctica vas a completar tú los huecos marcados con `# TODO:`. No hay celdas de solución ya rellenas: intenta resolver cada ejercicio antes de desplegar la solución colapsada que hay justo debajo del enunciado.

**Ejercicios:**
1. Cambiar `GlobalAveragePooling1D` por una capa `LSTM` en el clasificador de sarcasmo y comparar resultados
2. Generar texto con una semilla distinta y un número distinto de palabras en el modelo de poesía

## Preparación común

Ejecuta esta celda antes de empezar: descarga y prepara los datos de titulares sarcásticos que usarás en el ejercicio 1 (es el mismo preprocesado que en el notebook de teoría).

In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

!wget --no-check-certificate \
    https://storage.googleapis.com/learning-datasets/sarcasm.json \
    -O /tmp/sarcasm.json

vocab_size = 10000
embedding_dim = 16
max_length = 100
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"
training_size = 20000

with open("/tmp/sarcasm.json", 'r') as f:
    datastore = json.load(f)

sentences = [item['headline'] for item in datastore]
labels = [item['is_sarcastic'] for item in datastore]

training_sentences = sentences[0:training_size]
testing_sentences = sentences[training_size:]
training_labels = np.array(labels[0:training_size])
testing_labels = np.array(labels[training_size:])

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(training_sentences)

training_sequences = tokenizer.texts_to_sequences(training_sentences)
training_padded = pad_sequences(training_sequences, maxlen=max_length,
                                 padding=padding_type, truncating=trunc_type)

testing_sequences = tokenizer.texts_to_sequences(testing_sentences)
testing_padded = pad_sequences(testing_sequences, maxlen=max_length,
                                padding=padding_type, truncating=trunc_type)

print("Datos listos:", training_padded.shape, testing_padded.shape)

## Ejercicio 1 — De GlobalAveragePooling1D a LSTM

En el notebook de teoría, el clasificador de sarcasmo usaba esta arquitectura:

```
Embedding -> GlobalAveragePooling1D -> Dense(24, relu) -> Dense(1, sigmoid)
```

Tu tarea: sustituye la capa `GlobalAveragePooling1D` por una capa recurrente `tf.keras.layers.LSTM(32)`, entrena el modelo y compara la precisión de validación (`val_accuracy`) y el tiempo de entrenamiento con el modelo original.

**Pistas:**
- Una capa `LSTM` espera la secuencia completa de vectores de la capa `Embedding`; no necesitas aplanar ni promediar nada antes
- El resto de la arquitectura (las capas `Dense` finales, la función de pérdida, el optimizador) no cambia
- Entrena las mismas 10 épocas que en el notebook de teoría para que la comparación sea justa
- Fíjate también en `model.summary()`: ¿tiene más o menos parámetros que la versión con pooling?

In [ ]:
modelo_ejercicio1 = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    # TODO: sustituye la siguiente línea por una capa tf.keras.layers.LSTM(32)
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

modelo_ejercicio1.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
modelo_ejercicio1.summary()

# TODO: entrena el modelo 10 épocas con training_padded/training_labels y
# validation_data=(testing_padded, testing_labels), y guarda el resultado en
# una variable llamada historial_ejercicio1


<details>
<summary>💡 Ver solución</summary>

```python
modelo_ejercicio1 = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

modelo_ejercicio1.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
modelo_ejercicio1.summary()

historial_ejercicio1 = modelo_ejercicio1.fit(
    training_padded, training_labels,
    epochs=10,
    validation_data=(testing_padded, testing_labels),
    verbose=2
)

# Comparación: imprime la última val_accuracy para compararla con el modelo
# con GlobalAveragePooling1D del notebook de teoría
print("val_accuracy final (LSTM):", historial_ejercicio1.history['val_accuracy'][-1])
```

Con LSTM(32) el modelo tiene bastantes más parámetros que con `GlobalAveragePooling1D`
(la capa recurrente necesita pesos para las puertas de entrada, olvido y salida en
cada una de las 32 unidades) y cada época tarda más en entrenar. La precisión de
validación puede quedar parecida o incluso algo por debajo de la versión con
pooling: para detectar sarcasmo en titulares cortos, promediar los vectores de
las palabras ya captura buena parte de la señal, así que el coste extra de la
LSTM no siempre se traduce en mejor precisión. En textos más largos, donde el
orden y el contraste entre partes de la frase importan más, la diferencia suele
notarse más a favor de la LSTM.
</details>

## Ejercicio 2 — Generar poesía con otra semilla y otra longitud

Ahora vas a reutilizar el modelo de generación de texto entrenado con letras de canciones irlandesas. Primero ejecuta la celda de preparación (descarga los datos, los tokeniza y entrena el modelo). Después completa la función `generar_texto` y pruébala con una semilla (seed) y un número de palabras distintos a los del notebook de teoría.

**Pistas:**
- El bucle de generación repite tres pasos: convertir el texto actual en una secuencia de tokens, rellenarla con padding hasta `max_sequence_len - 1`, y usar el modelo para predecir el índice de la palabra más probable
- `np.argmax(model.predict(...), axis=-1)` te da el índice de la palabra predicha; tienes que buscar en `tokenizer.word_index` qué palabra corresponde a ese índice
- Prueba con una semilla que no aparezca en el corpus de canciones irlandesas (por ejemplo, una frase sobre tecnología) y compara la coherencia del resultado con una semilla más parecida al estilo del corpus

In [ ]:
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

!wget --no-check-certificate \
    https://storage.googleapis.com/learning-datasets/irish-lyrics-eof.txt \
    -O /tmp/irish-lyrics-eof.txt

poem_tokenizer = Tokenizer()
data = open('/tmp/irish-lyrics-eof.txt').read()
corpus = data.lower().split("\n")
poem_tokenizer.fit_on_texts(corpus)
total_words = len(poem_tokenizer.word_index) + 1

input_sequences = []
for line in corpus:
    token_list = poem_tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i + 1])

max_sequence_len = max(len(seq) for seq in input_sequences)
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

xs, labels_poem = input_sequences[:, :-1], input_sequences[:, -1]
ys = tf.keras.utils.to_categorical(labels_poem, num_classes=total_words)

poem_model = Sequential([
    Embedding(total_words, 64, input_length=max_sequence_len - 1),
    Bidirectional(LSTM(64)),
    Dense(total_words, activation='softmax')
])
poem_model.compile(loss='categorical_crossentropy',
                    optimizer=Adam(learning_rate=0.01),
                    metrics=['accuracy'])

poem_model.fit(xs, ys, epochs=30, verbose=1)
print("Modelo entrenado.")

In [ ]:
def generar_texto(seed_text, next_words, model, tokenizer, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        # TODO: rellena token_list con pad_sequences hasta maxlen=max_sequence_len - 1,
        # con padding='pre' (recuerda envolver token_list en una lista: [token_list])

        # TODO: usa model.predict sobre la secuencia rellenada y np.argmax(..., axis=-1)
        # para obtener el índice de la palabra predicha; guárdalo en la variable "predicted"

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

# TODO: prueba la función con una semilla distinta a la del notebook de teoría
# ("I've got a bad feeling about this") y con un número de palabras distinto a 20
nueva_semilla = ""
numero_de_palabras = 0
# print(generar_texto(nueva_semilla, numero_de_palabras, poem_model, poem_tokenizer, max_sequence_len))


<details>
<summary>💡 Ver solución</summary>

```python
def generar_texto(seed_text, next_words, model, tokenizer, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')

        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

nueva_semilla = "the new data center hummed with electricity"
numero_de_palabras = 40
print(generar_texto(nueva_semilla, numero_de_palabras, poem_model, poem_tokenizer, max_sequence_len))
```

Con una semilla ajena al corpus (por ejemplo, sobre centros de datos en lugar de
sobre Irlanda), el modelo intenta forzar el vocabulario y el estilo de las
canciones irlandesas sobre unas palabras iniciales que nunca vio en ese
contexto: el resultado suele perder coherencia bastante rápido. Cuantas más
palabras generes (`next_words` más alto), más se nota este efecto, porque cada
palabra nueva se genera a partir de las palabras anteriores ya generadas por el
propio modelo, y los pequeños errores se van acumulando.
</details>

## Para seguir practicando

- Sube el número de épocas del modelo de poesía (por ejemplo a 60 o 100, como en el tutorial original) y observa si el texto generado gana coherencia
- Prueba a apilar una segunda capa LSTM en el ejercicio 1 (recuerda `return_sequences=True` en la primera) y comprueba si mejora la precisión de validación
- Compara los tiempos de entrenamiento de las tres arquitecturas que has visto en esta clase: pooling, LSTM simple y LSTM bidireccional